In [ ]:
!pip install -U sentence-transformers pymilvus faiss-cpu pyarrow

In [ ]:
import pandas as pd
import numpy as np
import os

# 1. ĐỊNH NGHĨA ĐƯỜNG DẪN
# Trên Kaggle, nếu ông upload dataset lên thì nó thường nằm trong /kaggle/input/tên-dataset/
path_summary = '/kaggle/input/datasets/baubauuu/vnpaquet/VietNamText/milvus_summaryVN.parquet' 
path_alltext = '/kaggle/input/datasets/baubauuu/vnpaquet/VietNamText/milvus_chunksVN.parquet'

def check_kaggle_data(path, label):
    if os.path.exists(path):
        print(f" Đã tìm thấy file {label}")
        df = pd.read_parquet(path)
        
        print(f"--- Thông tin {label} ---")
        print(f"Số dòng: {len(df):,}")
        print("\n[Schema]:")
        print(df.dtypes)
        
        print("\n[Dữ liệu mẫu]:")
        # Hiển thị ngang cho đẹp trên Kaggle Cell
        display(df.head(3)) 
        print("="*50 + "\n")
        return df
    else:
        print(f" Không tìm thấy file {label} tại: {path}")
        # Liệt kê các file đang có trong input để ông check lại đường dẫn
        for dirname, _, filenames in os.walk('/kaggle/input'):
            for filename in filenames:
                print(f"File đang có: {os.path.join(dirname, filename)}")
        return None

# 2. CHẠY KIỂM TRA
df_summary = check_kaggle_data(path_summary, "TẦNG 1: SUMMARY")
df_alltext = check_kaggle_data(path_alltext, "TẦNG 2: ALL TEXT")

# 3. LOGIC TRUY XUẤT 2 TẦNG (MẪU)
def logic_goi_y(query_result_ids):
    """
    Giả sử sau khi search vector ở Tầng 1, ông có list IDs địa điểm.
    Hàm này sẽ lọc nhanh Tầng 2.
    """
    # query_result_ids là list các location_id từ tầng 1
    relevant_chunks = df_alltext[df_alltext['location_id'].isin(query_result_ids)]
    return relevant_chunks

print("Code đã sẵn sàng. Ông dán đường dẫn thực tế trên Kaggle vào rồi chạy nhé!")

In [ ]:
import torch
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder

# Kiểm tra GPU - Cực kỳ quan trọng để chạy bản Large
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f" Hệ thống đang chạy trên: {device}")

class ModelFactory:
    @staticmethod
    def get_encoder():
        # E5-Large: d=1024
        print(" Đang nạp Model E5-Large (Embedding)...")
        return SentenceTransformer('intfloat/multilingual-e5-large', device=device)
    
    @staticmethod
    def get_reranker():
        # BGE-Reranker: Thẩm định chuyên sâu
        print(" Đang nạp BGE-Reranker (Re-ranking)...")
        return CrossEncoder('BAAI/bge-reranker-base', device=device)

# Khởi tạo model dùng chung
encoder = ModelFactory.get_encoder()
reranker = ModelFactory.get_reranker()

In [4]:
import numpy as np
import faiss
import pandas as pd

class VectorEngine:
    """Mo phong mot Collection trong Milvus voi Index Faiss"""
    def __init__(self, df, name, vector_col='vector'):
        self.name = name
        self.df = df
        self.dim = 1024  # Khop voi E5-Large
        
        # Xay dung Index Faiss IndexFlatIP (Cosine Similarity sau khi normalize)
        print(f"Dang chi muc hoa du lieu cho {name}...")
        vectors = np.stack(df[vector_col].values).astype('float32')
        faiss.normalize_L2(vectors) 
        
        self.index = faiss.IndexFlatIP(self.dim)
        self.index.add(vectors)

    def search(self, query_vec, k=10, filter_ids=None, id_col='parent_id'):
        """
        Logic truy xuat:
        - Neu co filter_ids: Loc theo ID truoc, sau do moi search (Pre-filtering).
        - Neu khong: Search tren toan bo collection.
        """
        if filter_ids is not None:
            # Mo phong Pre-filtering
            candidate_df = self.df[self.df[id_col].isin(filter_ids)].copy()
            if candidate_df.empty: 
                return candidate_df
            
            c_vectors = np.stack(candidate_df['vector'].values).astype('float32')
            faiss.normalize_L2(c_vectors)
            
            # Tao index tam thoi cho tap du lieu da loc
            temp_index = faiss.IndexFlatIP(self.dim)
            temp_index.add(c_vectors)
            scores, indices = temp_index.search(query_vec, k)
            
            res = candidate_df.iloc[indices[0]].copy()
            res['score'] = scores[0]
            return res
        else:
            # Search toan bo index
            scores, indices = self.index.search(query_vec, k)
            res = self.df.iloc[indices[0]].copy()
            res['score'] = scores[0]
            return res

In [5]:
import time

class AdvancedRetrievalSystem:
    def __init__(self, encoder, reranker, summary_engine, chunk_engine):
        self.encoder = encoder
        self.reranker = reranker
        self.summary_engine = summary_engine
        self.chunk_engine = chunk_engine

    def invoke(self, 
               query_text: str, 
               use_expansion: bool = True, 
               use_summary: bool = True, 
               use_rerank: bool = True, 
               k_exp: int = 2,       # So luong dia danh dung de mo rong query
               k_summary: int = 10,  # So luong dia danh loc o Tang 1
               k_chunk: int = 20,    # So luong chunk lay o Tang 2
               top_n: int = 5        # Ket qua cuoi cung sau Re-rank
              ):
        
        start_time = time.time()
        
        # --- BUOC 1: PRE-RETRIEVAL (QUERY EXPANSION) ---
        queries = [query_text]
        if use_expansion:
            # Tham do nhanh de lay cac keyword (title) tu Summary
            q_vec_init = self.encoder.encode([f"query: {query_text}"], convert_to_numpy=True).astype('float32')
            faiss.normalize_L2(q_vec_init)
            
            # Dung k_exp de quy dinh so luong title muon mron
            quick_context = self.summary_engine.search(q_vec_init, k=k_exp)
            
            for _, row in quick_context.iterrows():
                # Tao cac bien the query moi kem theo ten dia danh
                queries.append(f"{query_text} {row['title']}") 
        
        # --- BUOC 2: EMBEDDING ALL QUERIES ---
        # Encode tat ca cac query (goc + mo rong) cung mot batch
        all_vecs = self.encoder.encode([f"query: {q}" for q in queries], convert_to_numpy=True).astype('float32')
        faiss.normalize_L2(all_vecs)
        main_vec = all_vecs[0].reshape(1, -1)

        # --- BUOC 3: RETRIEVAL LOGIC ---
        if use_summary:
            # CHE DO PHAN TANG: Hop nhat ID tu tat ca cac query mo rong
            all_parent_ids = set()
            for i in range(len(all_vecs)):
                res_l1 = self.summary_engine.search(all_vecs[i].reshape(1, -1), k=k_summary)
                all_parent_ids.update(res_l1['id'].tolist())
            
            if not all_parent_ids: 
                return pd.DataFrame(), {"status": "No IDs found", "latency_ms": 0}
            
            # Truy xuat trong pham vi cac Parent ID da loc
            raw_chunks = self.chunk_engine.search(main_vec, k=k_chunk, filter_ids=list(all_parent_ids))
        else:
            # CHE DO VET CAN: Search truc tiep tren toan bo kho Chunk
            raw_chunks = self.chunk_engine.search(main_vec, k=k_chunk, filter_ids=None)

        # --- BUOC 4: RE-RANKING ---
        if use_rerank and not raw_chunks.empty:
            pairs = [[query_text, c] for c in raw_chunks['content'].tolist()]
            raw_chunks['rerank_score'] = self.reranker.predict(pairs)
            final_df = raw_chunks.sort_values(by='rerank_score', ascending=False).head(top_n)
        else:
            final_df = raw_chunks.head(top_n)

        latency = (time.time() - start_time) * 1000
        return final_df, {
    "latency_ms": latency, 
    "mode": "Expansion-enabled" if use_expansion else "Standard",
    "queries_used": len(queries),
    "k_exp_applied": k_exp,
    "parent_id_count": len(all_parent_ids) if use_summary else 0  # Thêm dòng này
}

In [3]:
import json
import numpy as np

# 1. Định nghĩa đường dẫn file JSON của ông
queries_path = '/kaggle/input/datasets/baubauuu/testvn/benchmark_queries.json'

# 2. Đọc dữ liệu từ file[cite: 2]
with open(queries_path, 'r', encoding='utf-8') as f:
    benchmark_queries = json.load(f)

print(f" Đã nạp {len(benchmark_queries)} câu hỏi. Đang tiến hành embedding...")

# 3. Trích xuất danh sách câu hỏi[cite: 2]
queries = [item['query'] for item in benchmark_queries]

# 4. Thực hiện Embedding với prefix "query: " (bắt buộc cho E5)
# Sử dụng biến 'encoder' ông đã khởi tạo ở checkparquet.ipynb[cite: 3]
query_embeddings = encoder.encode([f"query: {q}" for q in queries], show_progress_bar=True, convert_to_numpy=True)

# 5. Lưu mảng vector xuống file .npy để dùng cho mọi bài test sau này
np.save('/kaggle/working/query_embeddings.npy', query_embeddings)

print(" Thành công! Đã lưu file tại: /kaggle/working/query_embeddings.npy")

 Đã nạp 1200 câu hỏi. Đang tiến hành embedding...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

 Thành công! Đã lưu file tại: /kaggle/working/query_embeddings.npy


In [ ]:
# 1. Load du lieu (Giu nguyen)
PATH_S = '/kaggle/input/datasets/baubauuu/vnpaquet/VietNamText/milvus_summaryVN.parquet'
PATH_C = '/kaggle/input/datasets/baubauuu/vnpaquet/VietNamText/milvus_chunksVN.parquet'

df_summary = pd.read_parquet(PATH_S)
df_chunks = pd.read_parquet(PATH_C)

# 2. Khoi tao Engine (Giu nguyen)
s_engine = VectorEngine(df_summary, "Summary_Layer")
c_engine = VectorEngine(df_chunks, "Chunk_Layer")

# Dung AdvancedRetrievalSystem de test linh hoat cac layer
system = AdvancedRetrievalSystem(encoder, reranker, s_engine, c_engine)

# 3. Dinh nghia cac to hop layer (Test Combinations)
test_combinations = [
    {
        "name": "FULL PIPELINE (Expansion + Summary + Rerank)",
        "params": {"use_expansion": True, "use_summary": True, "use_rerank": True}
    },
    {
        "name": "STANDARD TWO-TIER (Summary + Chunk + Rerank)",
        "params": {"use_expansion": False, "use_summary": True, "use_rerank": True}
    },
    {
        "name": "FAST SEARCH (Summary + Chunk - No Rerank)",
        "params": {"use_expansion": False, "use_summary": True, "use_rerank": False}
    },
    {
        "name": "BRUTE FORCE (Direct Chunk Search + Rerank)",
        "params": {"use_expansion": False, "use_summary": False, "use_rerank": True}
    }
]

query = "The historical development of Hai Phong port and its role in Vietnam's maritime economy"
get_name = lambda x: df_summary[df_summary['id'] == x]['title'].values[0]

print(f"QUERY TEST: {query}\n")

for case in test_combinations:
    print(f"=== TO HOP: {case['name']} ===")
    
    # Thuc thi va do toc do
    final_df, meta = system.invoke(
        query,
        k_summary=5,
        k_chunk=10,
        top_n=3,
        **case['params']
    )
    
    # Bao cao toc do va cau hinh
    print(f"Mode thuc thi: {meta.get('mode', 'N/A')}")
    print(f"Latency tong cong: {meta['latency_ms']:.2f} ms")
    if 'queries_used' in meta:
        print(f"So luong query (sau expansion): {meta['queries_used']}")
    
    # Hien thi du lieu doi chung
    if not final_df.empty:
        df_display = final_df.copy()
        
        # Mapping ten dia danh linh hoat theo id hoac parent_id
        id_col = 'parent_id' if 'parent_id' in df_display.columns else 'id'
        df_display['Location'] = df_display[id_col].apply(get_name)
        
        # Chon cac cot hien thi phu hop
        cols = ['Location', 'score', 'content']
        if 'rerank_score' in df_display.columns:
            cols.insert(1, 'rerank_score')
            
        display(df_display[cols])
    else:
        print("Trang thai: Khong tim thay ket qua.")
        
    print("=" * 80 + "\n")

In [ ]:
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import faiss
import time

def run_comprehensive_benchmark(system, queries_path, embedding_path, top_k_list=[1, 3, 5, 10], k_exp_test=5):
    """
    Ham benchmark toan dien 6 to hop layer.
    Tham so k_exp_test: So luong dia danh dung de tham do (expansion).
    """
    # 1. Load du lieu
    with open(queries_path, 'r', encoding='utf-8') as f:
        benchmark_queries = json.load(f)
    
    try:
        all_q_vecs = np.load(embedding_path)
        print(f"Da nap {len(all_q_vecs)} vectors tu file npy.")
    except Exception as e:
        print(f"Loi nap file embedding: {e}")
        return None

    # 2. DINH NGHIA 6 TO HOP THEO YEU CAU
    # k_exp chi co tac dung khi exp=True
    test_configs = test_configs = [
    # --- NHÓM 1: BASELINE & SIMPLE IMPROVEMENTS ---
    {"name": "1. Chunk (Baseline)", "exp": False, "sum": False, "rerank": False},
    {"name": "2. Chunk + Rerank", "exp": False, "sum": False, "rerank": True}, # MỚI: Chỉ Rerank
    {"name": "3. Exp + Chunk", "exp": True, "sum": False, "rerank": False},    # MỚI: Chỉ Exp

    # --- NHÓM 2: CÓ TẦNG LỌC (SUMMARY) ---
    {"name": "4. Sum + Chunk", "exp": False, "sum": True, "rerank": False},
    {"name": "5. Sum + Chunk + Rerank", "exp": False, "sum": True, "rerank": True},

    # --- NHÓM 3: CẤU HÌNH MẠNH (EXPANSION) ---
    {"name": "6. Exp + Chunk + Rerank", "exp": True, "sum": False, "rerank": True},
    {"name": "7. Exp + Sum + Chunk", "exp": True, "sum": True, "rerank": False},
    {"name": "8. Exp + Sum + Chunk + Rerank (Full)", "exp": True, "sum": True, "rerank": True}
]

    summary_results = []

    for config in test_configs:
        print(f"\n--- Dang chay Batch Test: {config['name']} ---")
        results = []
        latencies = []
        
        for i, item in enumerate(tqdm(benchmark_queries)):
            query_text = item['query']
            gt_content = str(item['chunk_text']).strip().lower() 
            
            # Thuc thi he thong voi tham so truyen vao tu config
            # k_summary=50 de dam bao do bao phu an toan cho tap du lieu lon
            final_df, meta = system.invoke(
                query_text,
                use_expansion=config['exp'],
                use_summary=config['sum'],
                use_rerank=config['rerank'],
                k_exp=k_exp_test if config['exp'] else 2, # Chi dung k_exp_test neu bat expansion
                k_summary=100, 
                k_chunk=10,
                top_n=max(top_k_list)
            )
            
            # Luu latency trung binh
            latencies.append(meta.get('latency_ms', 0))
            
            row = {'mrr': 0.0}
            for k in top_k_list: row[f'recall@{k}'] = 0.0
            
            if not final_df.empty:
                retrieved_contents = [str(c).strip().lower() for c in final_df['content'].tolist()]
                
                # Tinh MRR va Recall
                if gt_content in retrieved_contents:
                    rank = retrieved_contents.index(gt_content) + 1
                    row['mrr'] = 1.0 / rank
                    for k in top_k_list:
                        if rank <= k:
                            row[f'recall@{k}'] = 1.0
            
            results.append(row)
        
        # Tong hop metrics
        df_res = pd.DataFrame(results)
        metrics = ['mrr'] + [f'recall@{k}' for k in top_k_list]
        avg_metrics = df_res[metrics].mean().to_dict()
        
        avg_metrics['Architecture'] = config['name']
        avg_metrics['Avg_Latency_ms'] = np.mean(latencies)
        # Ghi chu k_exp thuc te da dung de de quan sat
        avg_metrics['K_Exp'] = k_exp_test if config['exp'] else "N/A"
        
        summary_results.append(avg_metrics)

    # 3. Xuat bao cao DataFrame
    report_df = pd.DataFrame(summary_results).set_index('Architecture')
    
    # Sap xep cot: Latency truoc, sau do den metrics
    cols = ['Avg_Latency_ms', 'K_Exp', 'mrr'] + [f'recall@{k}' for k in top_k_list]
    report_df = report_df[cols]
    
    print("\n" + "=" * 80)
    print(f"BAO CAO TOAN DIEN 6 TO HOP (K_EXPANSION = {k_exp_test})")
    print("=" * 80)
    display(report_df.round(4))
    
    return report_df

# --- CACH GOI DE TEST ---
# Ong co the truyen k_exp_test = 5 hoac 10 de thu nghiem
final_benchmark_report = run_comprehensive_benchmark(
    system, 
    QUERIES_FILE, 
    EMBED_FILE, 
    k_exp_test=2 
)

In [29]:
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import faiss
import time

def run_comprehensive_benchmark(system, queries_path, embedding_path, top_k_list=[1, 3, 5, 10], k_exp_test=5):
    """
    Ham benchmark toan dien 6 to hop layer.
    Tham so k_exp_test: So luong dia danh dung de tham do (expansion).
    """
    # 1. Load du lieu
    with open(queries_path, 'r', encoding='utf-8') as f:
        benchmark_queries = json.load(f)
    
    try:
        all_q_vecs = np.load(embedding_path)
        print(f"Da nap {len(all_q_vecs)} vectors tu file npy.")
    except Exception as e:
        print(f"Loi nap file embedding: {e}")
        return None

    # 2. DINH NGHIA 6 TO HOP THEO YEU CAU
    # k_exp chi co tac dung khi exp=True
    test_configs = test_configs = [
    # --- NHÓM 1: BASELINE & SIMPLE IMPROVEMENTS ---
    {"name": "1. Chunk (Baseline)", "exp": False, "sum": False, "rerank": False},
    {"name": "2. Chunk + Rerank", "exp": False, "sum": False, "rerank": True}, # MỚI: Chỉ Rerank
    {"name": "3. Exp + Chunk", "exp": True, "sum": False, "rerank": False},    # MỚI: Chỉ Exp

    # --- NHÓM 2: CÓ TẦNG LỌC (SUMMARY) ---
    {"name": "4. Sum + Chunk", "exp": False, "sum": True, "rerank": False},
    {"name": "5. Sum + Chunk + Rerank", "exp": False, "sum": True, "rerank": True},

    # --- NHÓM 3: CẤU HÌNH MẠNH (EXPANSION) ---
    {"name": "6. Exp + Chunk + Rerank", "exp": True, "sum": False, "rerank": True},
    {"name": "7. Exp + Sum + Chunk", "exp": True, "sum": True, "rerank": False},
    {"name": "8. Exp + Sum + Chunk + Rerank (Full)", "exp": True, "sum": True, "rerank": True}
]

    summary_results = []

    for config in test_configs:
        print(f"\n--- Dang chay Batch Test: {config['name']} ---")
        results = []
        latencies = []
        parent_counts = []
        
        for i, item in enumerate(tqdm(benchmark_queries)):
            query_text = item['query']
            gt_content = str(item['chunk_text']).strip().lower() 
            
            # Thuc thi he thong voi tham so truyen vao tu config
            # k_summary=50 de dam bao do bao phu an toan cho tap du lieu lon
            final_df, meta = system.invoke(
                query_text,
                use_expansion=config['exp'],
                use_summary=config['sum'],
                use_rerank=config['rerank'],
                k_exp=k_exp_test if config['exp'] else 2, # Chi dung k_exp_test neu bat expansion
                k_summary=100, 
                k_chunk=10,
                top_n=max(top_k_list)
            )
            
            # Luu latency trung binh
            latencies.append(meta.get('latency_ms', 0))
            parent_counts.append(meta.get('parent_id_count', 0))
            
            row = {'mrr': 0.0}
            for k in top_k_list: row[f'recall@{k}'] = 0.0
            
            if not final_df.empty:
                retrieved_contents = [str(c).strip().lower() for c in final_df['content'].tolist()]
                
                # Tinh MRR va Recall
                if gt_content in retrieved_contents:
                    rank = retrieved_contents.index(gt_content) + 1
                    row['mrr'] = 1.0 / rank
                    for k in top_k_list:
                        if rank <= k:
                            row[f'recall@{k}'] = 1.0
            
            results.append(row)
        
        # Tong hop metrics
        df_res = pd.DataFrame(results)
        metrics = ['mrr'] + [f'recall@{k}' for k in top_k_list]
        avg_metrics = df_res[metrics].mean().to_dict()
        
        avg_metrics['Architecture'] = config['name']
        avg_metrics['Avg_Latency_ms'] = np.mean(latencies)
        avg_metrics['Avg_Parent_IDs'] = np.mean(parent_counts)
        # Ghi chu k_exp thuc te da dung de de quan sat
        avg_metrics['K_Exp'] = k_exp_test if config['exp'] else "N/A"
        
        summary_results.append(avg_metrics)

    # 3. Xuat bao cao DataFrame
    report_df = pd.DataFrame(summary_results).set_index('Architecture')
    
    # Sap xep cot: Latency truoc, sau do den metrics
    cols = ['Avg_Latency_ms', 'Avg_Parent_IDs', 'K_Exp', 'mrr'] + [f'recall@{k}' for k in top_k_list]
    report_df = report_df[cols]
    
    print("\n" + "=" * 80)
    print(f"BAO CAO TOAN DIEN 6 TO HOP (K_EXPANSION = {k_exp_test})")
    print("=" * 80)
    display(report_df.round(4))
    
    return report_df

# --- CACH GOI DE TEST ---
# Ong co the truyen k_exp_test = 5 hoac 10 de thu nghiem
final_benchmark_report = run_comprehensive_benchmark(
    system, 
    QUERIES_FILE, 
    EMBED_FILE, 
    k_exp_test=3 
)

Da nap 1200 vectors tu file npy.

--- Dang chay Batch Test: 1. Chunk (Baseline) ---


  0%|          | 0/1200 [00:00<?, ?it/s]


--- Dang chay Batch Test: 2. Chunk + Rerank ---


  0%|          | 0/1200 [00:00<?, ?it/s]


--- Dang chay Batch Test: 3. Exp + Chunk ---


...

  0%|          | 0/1200 [00:00<?, ?it/s]


--- Dang chay Batch Test: 8. Exp + Sum + Chunk + Rerank (Full) ---


  0%|          | 0/1200 [00:00<?, ?it/s]


BAO CAO TOAN DIEN 6 TO HOP (K_EXPANSION = 3)


,Avg_Latency_ms,Avg_Parent_IDs,K_Exp,mrr,recall@1,recall@3,recall@5,recall@10
Architecture,,,,,,,,
1. Chunk (Baseline),19.7637,0.0,N/A,0.7643,0.6833,0.8300,0.8767,0.9050
2. Chunk + Rerank,149.0451,0.0,N/A,0.7927,0.7217,0.8592,0.8875,0.9050
3. Exp + Chunk,48.8900,0.0,3,0.7643,0.6833,0.8300,0.8767,0.9050
4. Sum + Chunk,21.9615,0.0,N/A,0.6542,0.5875,0.7125,0.7475,0.7617
5. Sum + Chunk + Rerank,151.2785,0.0,N/A,0.6649,0.6025,0.7225,0.7483,0.7617
6. Exp + Chunk + Rerank,180.8211,0.0,3,0.7927,0.7217,0.8592,0.8875,0.9050
7. Exp + Sum + Chunk,53.5774,0.0,3,0.6709,0.6025,0.7308,0.7658,0.7825
8. Exp + Sum + Chunk + Rerank (Full),185.7884,0.0,3,0.6827,0.6183,0.7425,0.7683,0.7825
